# Import Statements

In [1]:
import pandas as pd
df = pd.read_csv("2026dependenciesHWFile.csv")
# left_df = df[df.iloc[:6]]
left_df = df.iloc[:6]
right_df = df.iloc[8:10]
right_df

TypeError: Boolean array expected for the condition, not object

### Computing Courses

**Question 1**\
How many CIS courses at the 100 level are part of the Computer Information Systems, BS program?


In [2]:
df['courseNumber'] = df['courseNumber'].apply(str)

cis100_bs_df = df[
    (df['courseLetter'] == 'CIS') &
    (df['relatedTo'] == 'Computer Information Systems, BS') &
    (df['courseNumber'].str.startswith('1'))
    ]
cis100_bs_df.shape[0]

5

**Question 2**\
How many courses are offered with the CIS designation?

In [3]:
# cis_total_df = df[df.iloc[:, 9] == 'CIS']
# (df['courseLetter'] == 'AME').sum()
# cis_total_df.shape[0]
cis_total = (df['courseLetter'] == 'CIS').sum()
print(cis_total)

525


**Question 3**\
How many courses are offered with the CYB designation?

In [4]:
# cyb_total_df = df[df.iloc[:, 9] == 'CYB']
# cyb_total_df.shape[0]
cyb_total = (df['courseLetter'] == 'CYB').sum()
print(cyb_total)

41


**Question 4**\
How many courses are offered with the DSC designation?

In [5]:
# dsc_total_df = df[df.iloc[:, 9] == 'DSC']
# dsc_total_df.shape[0]
dsc_total = (df['courseLetter'] == 'DSC').sum()
print(dsc_total)

136


**Question 5**\
How many courses are offered with the ISS designation?

In [6]:
# iss_total_df = df[df.iloc[:, 9] == 'ISS']
# iss_total_df.shape[0]
iss_total = (df['courseLetter'] == 'ISS').sum()
print(iss_total)

179


### Crosslisting of Courses

**Create a dataframe with course designated as "CrossList"**

In [7]:
cl_df = df[df['How'] == 'CrossList']
cl_df.shape[0]

376

**Question 6**\
Find how many cross lists each course has.  How may courses are crosslisted with POS 223?

In [8]:
pos_223_cl_total = (cl_df['relatedTo'] == 'POS 223').sum()
print(pos_223_cl_total)

2


**Question 7**\
What is the mean number of crosslists per course?

In [9]:
cl_total = len(cl_df)
# groups courses by what they are related to and gives a count of how many courses have 1,2 or 3 crosslists
each_course_cl_count = cl_df.groupby('course')['relatedTo'].count()
one_cl = each_course_cl_count.value_counts()[1]
two_cl = each_course_cl_count.value_counts()[2] * 2
three_cl = each_course_cl_count.value_counts()[3] * 3
total_each_cl = one_cl + two_cl + three_cl
print(total_each_cl)
total_courses = df['course'].count()
print(total_each_cl / total_courses)
courses_total = df['courseDictionary'].count()
mean_cl_per_course = cl_total / courses_total
print(total_each_cl / courses_total)
print(round(mean_cl_per_course, 2))

# Create unique course identifier

df['full_course'] = (
    df.iloc[:, 1].astype(str).str.strip()
    + ' '
    + df.iloc[:, 2].astype(str).str.strip()
)
total_catalog_courses = df['full_course'].nunique()

# Mean across the full catalog
mean_catalog = 376 / total_catalog_courses
print(f'Catalog-wide mean: {mean_catalog:.4f}')

376
0.06174055829228243
0.2754578754578755
0.28
Catalog-wide mean: 0.3241


**Question 8**\
What is the median number of courses crosslisted?

In [10]:
non_cl_df = df[df['How'] != 'CrossList']
non_cl_series = pd.Series(0, index=non_cl_df.index)
cl_series = pd.Series(1, index=cl_df.index)

all_courses = pd.concat([non_cl_series, cl_series])
median = all_courses.median()

print(median)

0.0


**Question 9**\
What is the mode of the number of crosslisted courses?

In [11]:
mode = all_courses.mode().values[0]
print(mode)

0


**Question 10**\
Given that a course is actually crosslisted, what is the average number of courses it is crosslisted to?

In [12]:
course_cl = cl_df.groupby('course')['relatedTo'].count()
cl_mean = course_cl.mean()
print(round(cl_mean, 2))

1.39


**Question 11**\
How many courses have four designations (crosslisted designations count as 1 course in total)?  Note:  what would that look like in this dataset?

In [13]:
courses_with_four_designations = course_cl.value_counts()[3] / 4
print(int(courses_with_four_designations))

5


**Question 12**\
Assuming all of the crosslists are one course (e.g. CIS 255 and DSC 255 are one course), how many courses does UMA offer in its catalog?

In [14]:
two_designated_courses = course_cl.value_counts()[1]
three_designated_courses = course_cl.value_counts()[2]
four_designated_courses = course_cl.value_counts()[3]

total_cl_courses = two_designated_courses + three_designated_courses + four_designated_courses
total_courses_with_designations = (two_designated_courses / 2) + (three_designated_courses / 3) + (four_designated_courses / 4)

total_UMA_course_offerings = df["courseDictionary"].count() - (total_cl_courses - total_courses_with_designations)

print(int(total_UMA_course_offerings))

1214


**Question 13**\
All four of the designations CIS, CYB, DSC, and ISS are offered by the computing group.  How many specific courses are offered by the computing group.  Note:  courses crosslisted together are ONE course for this purpose.

In [15]:
comp_group_df = df[df['courseLetter'].isin(['CIS', 'CYB', 'DSC', 'ISS'])]
comp_group_cl_df = comp_group_df[comp_group_df['How'] == 'CrossList']
# all_comp_cl_df.shape[0]
total_comp_cl = comp_group_cl_df.groupby('course')['relatedTo'].count()
print(total_comp_cl.value_counts())
# with pd.option_context('display.max_rows', None, 'display.max_columns', None):  # more options can be specified also
#     print(comp_group_df)

relatedTo
1    18
2    10
3     2
Name: count, dtype: int64


### Architecture Program
In this section, we will be studying the courses and programs offered by the Architecture faculty.

**Question 14**\
What proportion of courses (by code – crosslists are separate for this purpose) are in the Architecture, B.Arch checksheet?

In [16]:
arch_checksheet_df = df[(df['relatedTo'] == 'Architecture, B.Arch')]
prop_barch = len(arch_checksheet_df) / courses_total
print(round(prop_barch, 3))

0.035


**Question 15**\
What proportion of courses (by code) are ARC courses?

In [17]:
# arc_total_df = df[df.iloc[:, 9] == 'ARC']
arc_total_df = df[df.iloc[:, 1] == 'ARC']
prop_arc = len(arc_total_df) / courses_total
print(round(prop_arc, 3))

0.084


**Question 16**\
Assuming these are independent, what proportion of courses (by code) are both in the Architecture, B.Arch checksheet and are ARC courses?

In [18]:
barch_arc_df = df[
    (df['relatedTo'] == 'Architecture, B.Arch') &
    (df['courseLetter'] == 'ARC')
    ]
prop_barch_arc = len(barch_arc_df) / courses_total
print(round(prop_barch_arc, 3))

0.021


**Question 17**\
What is the conditional probability that a given course with an ARC designation is part of the Architecture, B.Arch checksheet?

In [19]:
# To answer this, P(A|B) or the probability of A given that B has happened. A = total courses with ARC designation, and B = Architecture, B.Arch.
# cp_barch_arc = len(arc_total_df) / len(barch_arc_df)
print(len(arc_total_df))
print(len(barch_arc_df))
cp_barch_arc = len(barch_arc_df) / len(arc_total_df)
print(round(cp_barch_arc, 2))

115
29
0.25


**Question 18**\
What is the conditional probability that a given course in the Architecture, B.Arch checksheet is an ARC course?

In [20]:
# To answer this, P(A|B) or the probability of A given that B has happened. A = total courses with ARC designation, and B = Architecture, B.Arch. checksheet.
# cp_arc_barch = len(arc_total_df) / len(arch_checksheet_df)
print(len(arc_total_df))
print(len(arch_checksheet_df))
cp_arc_barch = len(arch_checksheet_df) / len(arc_total_df)
print(round(cp_arc_barch, 2))

115
48
0.42


**Question 19**\
Do the answers from 14-18 suggest that ARC courses and Architecture, B.Arch checksheet courses are independent statistically?  Why or why not.

### Prerequisites
In this section, we will address whether or not there is a bidirectional relationship over prerequisites (namely are courses equally likely to have prerequisites or to be prerequisites).

**Question 20**\
What proportion of courses (by code) are a prerequisite for another course?

In [21]:
prereq_total_df = df[
    (df['How'] == 'Prereq') &
    (df['relatedToCourseLetter'].notnull())
    ]

total_designations = df['course'].count()
print(len(prereq_total_df))
print(total_designations)
prereq_prop = len(prereq_total_df) / total_designations
print(round(prereq_prop, 2))

1722
6090
0.28


**Question 21**\
What proportion of courses (by code) have at least one prerequisite?

In [22]:
prereq_total = df[df['How'] == 'Prereq']
total_designations = df['course'].count()
print(len(prereq_total))
print(total_designations)
prereq_prop = len(prereq_total) / total_designations
print(round(prereq_prop, 2))

1724
6090
0.28


### American Studies
The Curriculum Committee has had a history of challenges with the American Studies program.  The coordinator of the program has what seems to be a liberal crosslisting policy.  This makes things difficult for those who maintain MaineStreet and maintain Acalog.  You will be addressing whether or not the data supports the claim of the Curriculum Committee that the coordinator of the program doth crosslist too much vis a vis their peers.

Recall:
An association rule has an antecedent and a consequent.  The antecedent begins the rule, while the consequent ends the rule.  If antecedent, then consequent.

**Support** is the proportion an item occurs in the dataset\
**Confidence** is the proportion of the time that given the antecedent, the consequent occurs.\
**Lift** is the ratio between the confidence of the rule and the support of its consequent.

**Question 22**\
What is the support for a course being an AME course?

In [23]:
ame_total = (df['courseLetter'] == 'AME').sum()
print(ame_total)
ame_support = ame_total / total_designations
print(round(ame_support, 3))

177
0.029


**Question 23**\
What is the support for a course being crosslisted?

In [24]:
total_cl = (df['How'] == 'CrossList').sum()
crossList_support = total_cl / total_designations
print(round(crossList_support, 3))

0.062


**Question 24**\
What is the confidence than an AME course is crosslisted?

In [25]:
# Given that course is AME it is crosslisted
# To answer this, P(A|B) or the probability of A given that B has happened. A = crosslisted, and B =AME course
# ame_conf = total_cl / ame_total
ame_cl = df[
    (df['How'] == 'CrossList') &
    (df['courseLetter'] == 'AME')
    ]
ame_conf = len(ame_cl) / ame_total
print(round(ame_conf, 2))

0.21


**Question 25**\
What is the lift associated with knowing that a course is an AME course?

In [26]:
# 2. Standardize prefix and unique course ID
# Using Column B (index 1) for primary courseLetter and Column C (index 2) for courseNumber
prefix = df.iloc[:, 1].astype(str).str.strip()
course_id = prefix + ' ' + df.iloc[:, 2].astype(str).str.strip()

# 3. Identify all crosslisted courses
# In this dataset, crosslist rows are marked in the 'How' or relationship column
type_col = 'How' if 'How' in df.columns else 'relationshipType'
cross_mask = df[type_col].astype(str).str.contains('cross', case=False, regex=False)

# Unique course codes that have at least one crosslist
crosslisted_courses = set(course_id[cross_mask].unique())

# 4. Define the universes
total_catalog_courses = course_id.nunique()
ame_mask = prefix == 'AME'
ame_courses = set(course_id[ame_mask].unique())

# 5. Calculate Support(B) = P(Crosslist)
support_b = len(crosslisted_courses) / total_catalog_courses

# 6. Calculate Confidence = P(Crosslist | AME)
ame_crosslisted = ame_courses.intersection(crosslisted_courses)
confidence = len(ame_crosslisted) / len(ame_courses) if len(ame_courses) > 0 else 0

# 7. Compute Lift
lift = confidence / support_b if support_b > 0 else 0

print(f"Total Unique Catalog Courses: {total_catalog_courses}")
print(f"Total Unique Crosslisted Courses: {len(crosslisted_courses)}")
print(f"Support(Crosslist) [P(B)]: {support_b:.4f}")
print(f"Total Unique AME Courses: {len(ame_courses)}")
print(f"AME Courses that are Crosslisted: {len(ame_crosslisted)}")
print(f"Confidence [P(Crosslist | AME)]: {confidence:.4f}")
print(f"Lift: {lift:.4f}")

Total Unique Catalog Courses: 1160
Total Unique Crosslisted Courses: 270
Support(Crosslist) [P(B)]: 0.2328
Total Unique AME Courses: 27
AME Courses that are Crosslisted: 22
Confidence [P(Crosslist | AME)]: 0.8148
Lift: 3.5007
